In [1]:
import os
os.environ["ARROW_USER_SIMD_LEVEL"] = "NONE"

import json, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report)
import joblib

SEED = 42
ART = os.path.expanduser("~/mlproject/artifacts")

print(f"sklearn: {sklearn.__version__}")
print(f"pandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")

sklearn: 1.9.1
pandas:  3.1.0
numpy:   2.4.6


In [2]:
df_tr = pd.read_parquet(f"{ART}/train.parquet")
df_va = pd.read_parquet(f"{ART}/val.parquet")
df_te = pd.read_parquet(f"{ART}/test.parquet")

def flatten_spans(df):
    rows = []
    for _, r in df.iterrows():
        text = r["text"] or ""
        starts = r["entity_starts"]
        if starts is None:
            continue
        try:
            if len(starts) == 0:
                continue
        except TypeError:
            continue
        for s, e, l in zip(r["entity_starts"], r["entity_ends"], r["entity_labels"]):
            rows.append({"span": text[s:e], "label": l})
    return pd.DataFrame(rows)

sp_tr = flatten_spans(df_tr)
sp_va = flatten_spans(df_va)
sp_te = flatten_spans(df_te)

print(f"спанов: train={len(sp_tr)}, val={len(sp_va)}, test={len(sp_te)}")
print("\nРаспределение типов в train:")
print(sp_tr["label"].value_counts())

спанов: train=24995, val=4346, test=3159

Распределение типов в train:
label
ip                            3751
name                          2365
first_name                    2358
date                          2112
address_city                  1861
last_name                     1650
mobile_phone                  1564
address                       1453
address_street                1407
document_number               1088
email                         1053
middle_name                    820
organization                   770
address_house                  620
address_geolocation            390
nickname                       275
name_initials                  246
vehicle_number                 239
address_district               216
snils                          187
military_individual_number     143
address_region                 116
tin                            101
address_country                 84
address_apartment               63
address_postal_code             38
address_build

In [3]:
majority = sp_tr["label"].value_counts().idxmax()
print(f"Самый частый тип в train: {majority} "
      f"({(sp_tr['label'] == majority).mean() * 100:.1f}% спанов)")

baseline_pred = np.full(len(sp_va), majority)
baseline_acc = accuracy_score(sp_va["label"], baseline_pred)
baseline_f1m = f1_score(sp_va["label"], baseline_pred, average="macro", zero_division=0)

print(f"\nBaseline на val:  accuracy={baseline_acc:.4f}, macro-F1={baseline_f1m:.4f}")

Самый частый тип в train: ip (15.0% спанов)

Baseline на val:  accuracy=0.1516, macro-F1=0.0098


In [5]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",       # символьные n-граммы ловят «форматные» признаки (email, IP, ИНН)
    ngram_range=(2, 5),
    min_df=1,
    max_features=10000,
    lowercase=True,
)

X_tr = vectorizer.fit_transform(sp_tr["span"])   # обучение словаря ТОЛЬКО на train
X_va = vectorizer.transform(sp_va["span"])       # та же матрица признаков
X_te = vectorizer.transform(sp_te["span"])

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",   # компенсация дисбаланса типов
    random_state=SEED,
)

t0 = time.time()
model.fit(X_tr, sp_tr["label"])
train_time = time.time() - t0
print(f"Обучение: {train_time:.2f} с, признаков: {X_tr.shape[1]}")

Обучение: 10.62 с, признаков: 10000


In [7]:
results = {}
for name, X, y in [
    ("val",  X_va, sp_va["label"]),
    ("test", X_te, sp_te["label"]),
]:
    pred = model.predict(X)
    acc = accuracy_score(y, pred)
    f1m = f1_score(y, pred, average="macro", zero_division=0)
    f1w = f1_score(y, pred, average="weighted", zero_division=0)
    results[name] = {"acc": float(acc), "macro_f1": float(f1m), "weighted_f1": float(f1w)}
    print(f"\n=== {name} ===")
    print(f"  accuracy:    {acc:.4f}")
    print(f"  macro-F1:    {f1m:.4f}")
    print(f"  weighted-F1: {f1w:.4f}")
    print(classification_report(y, pred, zero_division=0))


=== val ===
  accuracy:    0.8178
  macro-F1:    0.6583
  weighted-F1: 0.8226
                            precision    recall  f1-score   support

                   address       0.78      0.56      0.65       256
         address_apartment       0.12      0.25      0.17         8
          address_building       0.17      0.33      0.22         3
              address_city       0.79      0.76      0.77       355
           address_country       0.50      0.74      0.60        19
          address_district       0.34      0.63      0.44        35
       address_geolocation       0.86      0.99      0.92        68
             address_house       0.73      0.79      0.76        87
       address_postal_code       0.20      0.33      0.25         3
            address_region       0.37      0.62      0.46        29
            address_street       0.72      0.72      0.72       226
                      date       0.97      0.93      0.95       421
           document_number       0.8

In [8]:
joblib.dump(vectorizer, f"{ART}/lab2_span_tfidf.joblib")
joblib.dump(model,      f"{ART}/lab2_span_logreg.joblib")

report = {
    "task": "span-type classification (baseline component for NER)",
    "model": "LogisticRegression(class_weight='balanced')",
    "seed": SEED,
    "sklearn_version": sklearn.__version__,
    "train_time_sec": round(train_time, 3),
    "vectorizer": {"analyzer": "char_wb", "ngram_range": [2, 5],
                   "max_features": 10000, "min_df": 1},
    "n_features": int(X_tr.shape[1]),
    "sizes": {"train": len(sp_tr), "val": len(sp_va), "test": len(sp_te)},
    "baseline": {
        "majority_class": majority,
        "acc_val": float(baseline_acc),
        "macro_f1_val": float(baseline_f1m),
    },
    "model_metrics": results,
}
with open(f"{ART}/lab2_model_report.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\nСохранено:")
print(f"  {ART}/lab2_span_tfidf.joblib")
print(f"  {ART}/lab2_span_logreg.joblib")
print(f"  {ART}/lab2_model_report.json")
print(json.dumps(report, indent=2, ensure_ascii=False))


Сохранено:
  /home/mluser/mlproject/artifacts/lab2_span_tfidf.joblib
  /home/mluser/mlproject/artifacts/lab2_span_logreg.joblib
  /home/mluser/mlproject/artifacts/lab2_model_report.json
{
  "task": "span-type classification (baseline component for NER)",
  "model": "LogisticRegression(class_weight='balanced')",
  "seed": 42,
  "sklearn_version": "1.9.1",
  "train_time_sec": 10.619,
  "vectorizer": {
    "analyzer": "char_wb",
    "ngram_range": [
      2,
      5
    ],
    "max_features": 10000,
    "min_df": 1
  },
  "n_features": 10000,
  "sizes": {
    "train": 24995,
    "val": 4346,
    "test": 3159
  },
  "baseline": {
    "majority_class": "ip",
    "acc_val": 0.15163368614818223,
    "macro_f1_val": 0.009753209753209754
  },
  "model_metrics": {
    "val": {
      "acc": 0.8177634606534745,
      "macro_f1": 0.6582896339388128,
      "weighted_f1": 0.822627736643917
    },
    "test": {
      "acc": 0.8088002532446977,
      "macro_f1": 0.6306346060388888,
      "weighted_f1"